# 批量推理

上一章，我们利用**键值缓存**加速了 GPT 模型的生成效率。本章，我们探讨一下从另一个角度提升模型生成效率：**并行**。

实际的生产环境中，GPT 模型经常需要同时处理多个生成请求。比如一个聊天机器人，同时有 100 个用户在线提问。如果逐个处理，用户需要排队等待；如果可以并行批量处理，则可以一次计算所有请求，大大提高吞吐量。

## 批量推理

**批量推理**（Batch Inference）的核心思想：把多个请求打包成一个批次，一次前向传播完成所有请求的计算。

### 实现原理

* **填充**：不同的请求的提示长度不同，因此要用特殊标记（比如：0）吧短的序列补齐到批次内最大的提示长度；
* **掩码**：每个请求的生成文本长度也不同，因此需要用掩码标记哪些位置时有效词元，哪些是填充。

In [1]:
import time
from pathlib import Path

import requests

from src.core import *
from src.gpt import *

In [2]:
np.random.seed(42)

## 模型

### GPT 批量嵌入层

**GPT 批量嵌入层**需要支持自定义的位置编码：每个序列的位置独立编码，不受填充影响。

In [3]:
class GPTEmbeddingWithPadding(GPTEmbedding):

    def forward(self, x: Tensor, positions):
        token = self.embedding(x)
        position = self.positional_embedding(Tensor(positions))
        return self.dropout(token + position)

### GPT 批量注意力层

**GPT 批量注意力层**需要处理填充掩码：

* 计算注意力得分后，吧填充位置的得分设置为 $-\infty$；
* Softmax 之后，填充位置的权重为 0，不影响有效词元。

In [4]:
class GPTAttentionWithPadding(GPTAttention):

    def forward(self, x: Tensor, valid):
        norm = self.normalize(x)
        head_embedding_size = self.embedding_size // self.heads
        multi_head_shape = (-1, x.shape[1], self.heads, head_embedding_size)
        shape = (-1, x.shape[1], self.embedding_size)

        query = self.query(norm).reshape(multi_head_shape).transpose((0, 2, 1, 3))
        key = self.key(norm).reshape(multi_head_shape).transpose((0, 2, 3, 1))
        value = self.value(norm).reshape(multi_head_shape).transpose((0, 2, 1, 3))
        scale = Tensor(np.array(1.0 / np.sqrt(head_embedding_size)))
        scores = self.mask(query @ key * scale)

        scores = Tensor(np.where(valid[:, None, None, :], scores.data, self.mask.value))
        weights = self.softmax(scores)
        return x + self.dropout(self.output((weights @ value).transpose((0, 2, 1, 3)).reshape(shape)))

### GPT 批量转换器模块

**GPT 批量转换器模块**传递掩码参数，确保填充位置不参与计算。

In [5]:
class GPTTransformerWithPadding(Composite):

    def __init__(self, embedding_size, heads):
        self.attention = GPTAttentionWithPadding(embedding_size, heads)
        self.feed_forward = GPTFeedForward(embedding_size)

        super().__init__([self.attention,
                          self.feed_forward])

    def forward(self, x: Tensor, valid):
        return self.feed_forward(self.attention(x, valid))

### GPT 批量模型

**GPT 批量模型**接收掩码和位置编码，逐层传递。

In [6]:
class GPTWithPadding(Composite):

    def __init__(self, vocab_size, context_size, embedding_size, heads, blocks):
        self.embedding = GPTEmbeddingWithPadding(vocab_size, context_size, embedding_size)
        self.transformers = [GPTTransformerWithPadding(embedding_size, heads) for _ in range(blocks)]
        self.output = GPTOutput(embedding_size, vocab_size)

        super().__init__([self.embedding] + self.transformers + [self.output])

    def forward(self, x: Tensor, valid, positions):
        x = self.embedding(x, positions)
        for layer in self.transformers:
            x = layer(x, valid)
        return self.output(x)

## GPT 批量训练器

**GPT 批量训练器** GPTBatchTrainer 的 generate 函数：

* **批量输入**：同时处理多个提示；
* **动态填充**：每步生成前，把当前序列填充到批次内最大长度；
* **独立采样**：每个序列独立采样生成下一个词元。

In [7]:
class GPTBatchTrainer(GPTTrainer):

    def generate(self, dataset, prompts, steps=256):
        self.model.eval()
        batch_tokens = [dataset.encode(p) for p in prompts]
        n = len(batch_tokens)

        with Tensor.no_grad():
            for _ in range(steps):
                windows = [tokens[-dataset.context_size:] for tokens in batch_tokens]
                width = max(len(w) for w in windows)

                features = np.zeros((n, width), dtype=np.int64)
                valid = np.zeros((n, width), dtype=bool)
                positions = np.zeros((n, width), dtype=np.int64)
                for i, window in enumerate(windows):
                    pad = width - len(window)
                    features[i, pad:] = window
                    valid[i, pad:] = True
                    positions[i, pad:] = np.arange(len(window))

                logits = self.model(Tensor(features), valid, positions)

                for i, tokens in enumerate(batch_tokens):
                    last_logits = logits.data[i, -1]
                    exp = np.exp(last_logits - np.max(last_logits))
                    probs = exp / np.sum(exp)
                    token = np.random.choice(len(probs), p=probs)
                    tokens.append(token)
        return [dataset.decode(t) for t in batch_tokens]

## 超参数

### 模型参数

模型参数需要保持和预训练阶段一致。

In [8]:
BATCH_SIZE = 16
CONTEXT_SIZE = 256
EMBEDDING_SIZE = 384
HEADS = 6
BLOCKS = 6

## 建模

### 加载模型

In [9]:
DATA_FILE = "../../../../data/tinyshakespeare.txt"
MODEL_FILE = "../../../../data/tinyshakespeare-gpt.npz"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPTWithPadding(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(model.parameters)
trainer = GPTBatchTrainer(model, loss_fn, optimizer)
trainer.load(MODEL_FILE)

## 批量提示词

In [10]:
prompts = ["ROMEO:", "JULIET:", "HAMLET:", "MACBETH:"]

## 串行推理

In [11]:
np.random.seed(42)

start = time.time()
for p in prompts:
    print(trainer.generate(dataset, prompts=[p])[0])
print(f"{time.time() - start:.8f} seconds")

ROMEO:
Why day, God be me, and then of you?

KING HENRY VI:
On a heauth, seven to the speak of men.

KING EDWARD IV:
But will to Richard above thy dear of creature.

KING RICHARD II:
Know then, let's golding at Capalies their might,
It she come by that you have 
JULIET:
That, is the strongest the man, made condemn'd to great
from her show'd for a brother in saints,
And when my woman and tame sport cotuned worship,
From Lewis help-robb in: O so and saint
Too have valamage, my graves towards requidest,
Being wrong. But, no
HAMLET:
We have how me as yourself your gates, mishadows a
heart of sures counsel about his dame call her
To that see his parts land dear hath so little poor!

CAMILLO:
It is, behold!

First Gentleman:
Why, let's counsel; the speak so.

POLIXENES:
I shall can pro
MACBETH:
Thou seest the crowned no trough of time women,
I go glad is dead, which at inviolent day:
The jurious of Norfolk and death.

CATESBY:
It is not on victory,
For Love battle on Barting that
But dead.



## 批量推理

In [12]:
np.random.seed(42)

start = time.time()
results = trainer.generate(dataset, prompts=prompts)
for r in results:
    print(r)
print(f"{time.time() - start:.8f} seconds")

ROMEO:
But she hallone a stern,
Whose Hastings a more to partain holy virtuous
To stands I like it as to thee:
The absences a nire sproudd your kingdom of the majesty
As did a sun following himself
Figure with proudlease this dog: and his war
Unnerish shame, wea
JULIET:
But I am life between the breast of gother hate.

ANGELO:
Ay, found: farewell; regarest the parties; I hear
duty.

HORTENSIO:
To king once him voice!

PAULINA:
Why, can goes awaking my tongue?

KING HENRY VI:
O honour, no; tell up his angel; after and I g
HAMLET:
A chambin hot this folling others living power him
In attenment, or sin, my lord.

KING RICHARD III:
His in sixtain Boubbels to your heart.

QUEEN MARGARET:
And I shall be be him a debraced rather Franks
Did night me a both and succeed of grief stay?

QUE
MACBETH:
Treason; whose shore recounted foul deny I croward.

QUEEN ELIZABETH:
March and oscench, loaths it tapply.

CAPULET:
You methink your now oaths far a house.

JULIET:
Would shall not we must to see you

本章，我们实现了**批量推理**：同时处理多个生成请求，通过填充和掩码对其序列，一次前向传播完成所有计算。

但是在上面的推理中我们可以看到，批量推理并没有表现出效率提高。这主要有两个原因：

* **总计算量没有减少**：我们没有使用 KV 缓存，每一步都要对整个窗口重新计算。批量只是把多个请求合并成更大的矩阵乘法，总的计算量与串行完全相同；
* **没有闲置的算力可以利用**：NumPy 的矩阵乘法本身就是多线程的，单个请求已经能够占满 CPU。在支持 GPU 的设备上，批量推理的收益会更加显著。

至此，我们完成了**模型推理**，一步步解决了**控制生成**和**提高效率**的和姓问题：

* **采样策略**：控制生成的质量和多样性；
* **停止条件**：控制生成的终止时机；
* **键值缓存**：加速自回归生成；
* **批量推理**：提高模型吞吐量。

## 课后练习

布置一个大作业：把**采样策略**、**停止条件**、**键值缓存**和**批量推理**集成到一起。